In [143]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from torch.utils.data import Dataset , DataLoader , random_split ,Subset

In [9]:
df = pd.read_csv(r"C:\Self Learning\Pytorch\RNN\100_Unique_QA_Dataset.csv")
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [ ]:
# Tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace("'","")
    text = text.replace("?","")
    return text.split()


['what', 'is', 'the', 'capital', 'of', 'france']

In [74]:
question_vocab = {"<NULL>":0}
answer_vocab = {"<NULL>":0}
def build_vocab(row):
    question = row['question']
    answer = row['answer']
    tokenized_question = tokenize(question)
    tokenized_answer = tokenize(answer)
    for token in tokenized_question:
        if token not in question_vocab:
            question_vocab[token] = len(question_vocab)
    for token in tokenized_answer:
        if token not in answer_vocab:
            answer_vocab[token] = len(answer_vocab)
df.apply(build_vocab , axis=1)



0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [76]:
count = 0
for key , value in answer_vocab.items():
    print(key ," : ",value)
    count+=1
    if count == 10:
        break

<NULL>  :  0
paris  :  1
berlin  :  2
harper-lee  :  3
jupiter  :  4
100  :  5
leonardo-da-vinci  :  6
8  :  7
au  :  8
1945  :  9


In [145]:
def text_index(text , vocab):
    tokenized_text = tokenize(text)
    index = []
    for word in tokenized_text:
        if word in vocab:
            index.append(vocab[word])
        else:
            index.append(vocab["<NULL>"])
    return index

class TextDataSet(Dataset):
    def __init__(self , df , question_vocab , answer_vocab):
        self.df = df
        self.question_vocab = question_vocab
        self.answer_vocab = answer_vocab
    def __len__(self):
        return len(df)
    def __getitem__(self, index):
        question = text_index(df.iloc[index]['question'] , self.question_vocab)
        answer = text_index(df.iloc[index]['answer'] , self.answer_vocab)
        return torch.tensor(question) , torch.tensor(answer)

              

In [281]:

dataset = TextDataSet(df , answer_vocab=answer_vocab , question_vocab=question_vocab)
train_data = DataLoader(dataset , batch_size=1 ,shuffle=True)

In [253]:
import random
idx = random.randint(0,len(dataset))
x, y = dataset[idx]
print(x,y)

tensor([ 34, 161,  89, 162, 163,  16,  12, 164,  35]) tensor([58])


# Main code

In [254]:
question_em = dataset[0][0]
answer_em = dataset[0][1]
print(question_em.shape)
print(answer_em.shape)

torch.Size([6])
torch.Size([1])


In [255]:
x = nn.Embedding(len(question_vocab) , 50)
# create uiformly distributed random numbers for each 
# index in the question vocab and each index will be represented by a 50 dimensional vector 
# x.weight.shape 
a = x(question_em)
a.shape

torch.Size([6, 50])

In [256]:
y = nn.RNN(50 , 64 ,batch_first=True)
rnn_out = y(a)
final_out = rnn_out[1]
final_out.shape

torch.Size([1, 64])

In [257]:
dense = nn.Linear(64 , len(answer_vocab))
dense_out = dense(final_out)
dense_out.shape

torch.Size([1, 86])

In [278]:
class RNNQA_Model(nn.Module):
    def __init__(self , vocab_size , answer_vocab_size):
        super().__init__()
        self.em_layer = nn.Embedding(vocab_size, 100)
        self.rnn = nn.RNN(100 , 64 ,batch_first=True)
        self.dense = nn.Linear(64 , answer_vocab_size)
        self.ac1 = nn.Sigmoid()
    def forward(self , x):
        x = self.em_layer(x)
        x = self.rnn(x)[1]
        x = self.ac1(x)
        out = self.dense(x)
        return out

In [284]:
lr = 0.001
model = RNNQA_Model(len(question_vocab) , len(answer_vocab))
optimizer = optim.Adam(model.parameters() , lr=lr)
# scheduler = optim.lr_scheduler.StepLR(optimizer , gamma=0.1 , step_size=10)
loss_fn = nn.CrossEntropyLoss()


In [286]:
def calculate_accuracy(y_pred , y_true):
    _, predicted = torch.max(y_pred, dim=1)
    correct = (predicted == y_true).sum().item()
    total = y_true.size(0)
    accuracy = correct / total
    return accuracy


for epoch in range(50):
    train_loss = 0.0
    train_accuracy = 0.0
    for X, y in train_data:

        y = y.flatten().long() 
        y_pred = model(X) 
        # print(y_pred.argmax(dim=2))          
        y_pred = y_pred.squeeze(1)
        loss = loss_fn(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_accuracy += calculate_accuracy(y_pred, y)
    print(f"Epoch {epoch+1} , Loss: {train_loss/len(train_data):.4f}, Accuracy: {train_accuracy/len(train_data):.4f}")

        

Epoch 1 , Loss: 2.5620, Accuracy: 0.9444
Epoch 2 , Loss: 2.4953, Accuracy: 0.9333
Epoch 3 , Loss: 2.4314, Accuracy: 0.9556
Epoch 4 , Loss: 2.3675, Accuracy: 0.9556
Epoch 5 , Loss: 2.3082, Accuracy: 0.9556
Epoch 6 , Loss: 2.2493, Accuracy: 0.9556
Epoch 7 , Loss: 2.1891, Accuracy: 0.9667
Epoch 8 , Loss: 2.1275, Accuracy: 0.9778
Epoch 9 , Loss: 2.0729, Accuracy: 0.9778
Epoch 10 , Loss: 2.0115, Accuracy: 0.9556
Epoch 11 , Loss: 1.9586, Accuracy: 0.9667
Epoch 12 , Loss: 1.9024, Accuracy: 0.9556
Epoch 13 , Loss: 1.8472, Accuracy: 0.9667
Epoch 14 , Loss: 1.7935, Accuracy: 0.9889
Epoch 15 , Loss: 1.7370, Accuracy: 0.9889
Epoch 16 , Loss: 1.6858, Accuracy: 0.9889
Epoch 17 , Loss: 1.6350, Accuracy: 0.9778
Epoch 18 , Loss: 1.5836, Accuracy: 0.9778
Epoch 19 , Loss: 1.5361, Accuracy: 0.9667
Epoch 20 , Loss: 1.4892, Accuracy: 0.9778
Epoch 21 , Loss: 1.4405, Accuracy: 0.9889
Epoch 22 , Loss: 1.3949, Accuracy: 1.0000
Epoch 23 , Loss: 1.3491, Accuracy: 1.0000
Epoch 24 , Loss: 1.3057, Accuracy: 0.9889
E

In [261]:
def idx_to_word(vocab):
    new_dic = dict()
    for key , value in vocab.items():
        new_dic[value] = key
    return new_dic
idx_to_question_vocab = idx_to_word(question_vocab)
idx_to_answer_vocab = idx_to_word(answer_vocab)


In [262]:
def idx_to_question(indices , idx_vocab):
    sentences = "" 
    for idx in indices:
        sentences += idx_vocab[int(idx)] + " "

    return sentences 

# Evaluation of the model

In [287]:
count = 0
for X , y in train_data:
    q_idices = X.clone().squeeze()
    a_indices = y
    model.eval()
    y_pred = model(X)
    y_pred = torch.argmax(y_pred, dim=2)
    question = idx_to_question(q_idices , idx_to_question_vocab)
    answer = idx_to_question(a_indices , idx_to_answer_vocab)
    predicted_answer = idx_to_question(y_pred , idx_to_answer_vocab)
    if answer == predicted_answer:
        count +=1
    print(question , "| Predicted Answer is : ", predicted_answer , "| True Answer : ",answer)


which planet is the closest to the sun  | Predicted Answer is :  mercury  | True Answer :  mercury 
who was the first person to step on the moon  | Predicted Answer is :  armstrong  | True Answer :  armstrong 
what is the capital of canada  | Predicted Answer is :  ottawa  | True Answer :  ottawa 
how many legs does a spider have  | Predicted Answer is :  8  | True Answer :  8 
which country is home to the great wall  | Predicted Answer is :  china  | True Answer :  china 
what is the capital of south korea  | Predicted Answer is :  seoul  | True Answer :  seoul 
what is the capital of india  | Predicted Answer is :  delhi  | True Answer :  delhi 
who painted the mona lisa  | Predicted Answer is :  leonardo-da-vinci  | True Answer :  leonardo-da-vinci 
what is the smallest prime number  | Predicted Answer is :  2  | True Answer :  2 
who discovered gravity  | Predicted Answer is :  newton  | True Answer :  newton 
who wrote to kill a mockingbird  | Predicted Answer is :  harper-lee  | 

In [288]:
print("Accuracy is : ",(len(train_data) / count) * 100,"%")

Accuracy is :  100.0 %
